# Transform contours i/o demo

Transform contours are timeseries with associated metadata. In this demo we'll generate an example, and save/load it to show how the interface works.

In [1]:
from os import path
from pathlib import Path

import numpy as np
from numpy.random import rand

from kymata.entities.contours import ContourSet
from kymata.io.contours import save_contours, load_contours

## Creating and using a `ContourSet` object

We'll use the example of TVL, and generate three totally random contours, as if sampled at 500 Hz for 10 seconds:

In [2]:
tvl_contours = ContourSet(
    {"IL1": rand(5_000),
     "IL2": rand(5_000),
     "IL3": rand(5_000),
     "IL4": rand(5_000),
     "IL":  rand(5_000)},
    description="TVL",
    sample_rate_hz=500,
)

The contours object will keep track of its own length

In [3]:
print(tvl_contours.duration_seconds)

10.0


Add new contours to the object:

In [4]:
tvl_contours.add_contour("STL", rand(5_000))
print(tvl_contours.names)

['IL1', 'IL2', 'IL3', 'IL4', 'IL', 'STL']


## Save and load

We'll save and reload the object to demonstrate the file format

In [5]:
sample_data_dir = Path(Path(path.abspath("")).parent, "kymata-core-data")
sample_data_dir.mkdir(exist_ok=True)

try:
    save_contours(tvl_contours, sample_data_dir / "tvl.contours")
    created_contours = True
except FileExistsError:
    print("Skipped")
    created_contours = False
print(list(sample_data_dir.glob("*.contours")))

[PosixPath('/Users/cai/Dox/Work/Kymata lab/Code/kymata-core/kymata-core-data/tvl.contours')]


Now reload the same object

In [6]:
tvl_reloaded = load_contours(sample_data_dir / "tvl.contours")
print(tvl_reloaded.names)
print(tvl_reloaded.duration_seconds)

['IL1', 'IL2', 'IL3', 'IL4', 'IL', 'STL']
10.0


## Load other formats

We can also load just plan `.npz`/`.npy` files:

In [7]:
if (sample_data_dir / "tvl.npz").exists():
    created_npz = False
else:
    np.savez(
        sample_data_dir / "tvl.npz",
        IL1=rand(5_000),
        IL2=rand(5_000),
        IL3=rand(5_000),
        IL4=rand(5_000),
        IL= rand(5_000),
        STL=rand(5_000),
    )
    created_npz = True

However in this case, the metadata won't have been saved with the file, so we'll need to supply it ourselves

In [8]:
tvl_from_npz = load_contours(sample_data_dir / "tvl.npz")
print(tvl_from_npz.names)
print(tvl_from_npz.sample_rate_hz)
print(tvl_from_npz.duration_seconds)

['IL1', 'IL2', 'IL3', 'IL4', 'IL', 'STL']
1000
5.0


In [9]:
tvl_from_npz = load_contours(sample_data_dir / "tvl.npz", replace_sample_rate=500)
print(tvl_from_npz.sample_rate_hz)
print(tvl_from_npz.duration_seconds)

500
10.0


And cleanup

In [10]:
if created_contours:
    (sample_data_dir / "tvl.contours").unlink()
if created_npz:
    (sample_data_dir / "tvl.npz").unlink()